# Workstream 3: Label And Music Continuity EDA

正解 track が会話内、過去 user session、artist / album / tag continuity のどの signal に寄っているかを確認します。

主な既存成果物: `relisten_rates_by_*.csv`, `relisten_actionability.csv`, `EDA/summary/relisten-eda-summary.md`.


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = Markdown = display = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "EDA").exists():
            return candidate
    raise RuntimeError("Repository root was not found from the current working directory.")

ROOT = find_repo_root(Path.cwd())
EDA_DIR = ROOT / "EDA"
TABLE_DIR = EDA_DIR / "tables"
FIGURE_DIR = EDA_DIR / "figures"
SUMMARY_DIR = EDA_DIR / "summary"
EXPERIMENT_DIR = ROOT / "mcrs" / "experiments"
INFERENCE_DIR = ROOT / "exp" / "inference"

for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"TABLE_DIR={TABLE_DIR}")


In [ ]:
def read_table(name: str, **kwargs) -> pd.DataFrame:
    path = TABLE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_csv_path(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if df.empty:
        print("empty dataframe")
        return
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def show_image(name: str) -> None:
    path = FIGURE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return
    if display is not None and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(path)


def save_table(df: pd.DataFrame, name: str) -> Path | None:
    if df.empty:
        print(f"skip empty table: {name}")
        return None
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(ROOT)} ({len(df):,} rows)")
    return path


def barplot(df: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = "", rotate: int = 0, figsize=(10, 4)) -> None:
    if df.empty:
        print("skip empty plot")
        return
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        df.plot(kind="bar", x=x, y=y, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    plt.show()


## Load Existing Continuity Tables


In [ ]:
by_split = read_table("relisten_rates_by_split.csv")
by_turn = read_table("relisten_rates_by_turn.csv")
by_goal = read_table("relisten_rates_by_goal.csv")
by_user_split = read_table("relisten_rates_by_user_split.csv")
by_history_size = read_table("relisten_rates_by_history_size.csv")
actionability = read_table("relisten_actionability.csv")

for name, df in {
    "by_split": by_split,
    "by_turn": by_turn,
    "by_goal": by_goal,
    "by_user_split": by_user_split,
    "by_history_size": by_history_size,
    "actionability": actionability,
}.items():
    print(f"\n{name}: {df.shape}")
    show_df(df)


## Overall Continuity Rates


In [ ]:
if not by_split.empty:
    pivot = by_split.pivot_table(index=["split", "scope"], columns="repeat_type", values="rate", aggfunc="first").reset_index()
    show_df(pivot, 50)
    plot_df = by_split[by_split["repeat_type"].isin(["exact", "artist", "album", "tag"])]
    barplot(plot_df, x="scope", y="rate", hue="repeat_type", title="Continuity rate by scope", figsize=(10, 4))

show_image("relisten_by_turn.png")
show_image("relisten_by_goal.png")
show_image("relisten_history_size.png")


## Turn / Goal / User Split Buckets


In [ ]:
if not by_turn.empty:
    fig, ax = plt.subplots(figsize=(11, 4))
    plot_df = by_turn[(by_turn["scope"] == "combined") & (by_turn["repeat_type"].isin(["artist", "album", "tag", "exact"]))]
    if sns is not None:
        sns.lineplot(data=plot_df, x="turn_number", y="rate", hue="repeat_type", style="split", marker="o", ax=ax)
    ax.set_title("Combined continuity rate by target turn")
    fig.tight_layout()
    plt.show()

if not by_user_split.empty:
    plot_df = by_user_split[(by_user_split["scope"] == "user") & (by_user_split["repeat_type"].isin(["exact", "artist", "album"]))]
    barplot(plot_df, x="user_split", y="eligible_rate", hue="repeat_type", title="User-history continuity by user split", rotate=30, figsize=(10, 4))


## Actionability As Candidate Source


In [ ]:
if not actionability.empty:
    summary = actionability.pivot_table(
        index=["split", "scope", "source", "topk"],
        values=["active_rate", "recall", "eligible_recall"],
        aggfunc="first",
    ).reset_index().sort_values(["split", "scope", "source", "topk"])
    show_df(summary, 100)

    plot_df = actionability[actionability["topk"].isin([20, 50, 100])]
    fig, ax = plt.subplots(figsize=(12, 5))
    if sns is not None:
        sns.lineplot(data=plot_df, x="topk", y="eligible_recall", hue="source", style="scope", marker="o", ax=ax)
    ax.set_title("Prior source eligible recall by topK")
    fig.tight_layout()
    plt.show()


## Findings / Decisions / Next Actions

- Findings:
- Decisions:
- Next actions:
